# 111장 · 최소 재현 입력과 회귀 테스트

최소 재현 입력은 결함을 드러내는 가장 작은 사례입니다. 고친 뒤에는 실패 사례뿐 아니라 원래 정상인 입력의 결과도 확인합니다.

장비명 정규화 수정 후 원래 정상 질문, 실패 재현 입력, 장비AB 반례를 다시 검사합니다. 릴리스 후보에 채택 범위와 회귀 결과를 기록합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 캐시와 표기 수정의 회귀 결과를 묶기

110장에서 변경 전후 원문이 같았는지 읽고, 98장에서 남긴 입력을 후보 정규화 함수에 다시 넣습니다. 둘 중 하나라도 실패하면 후보를 보류합니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter110.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter111-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter110.json"
sample_file = here / "chapter111-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['cache_check'])

교재 예시 chapter111-start.json
{'same': True, 'hits': 1, 'misses': 1}


두 번째 줄은 읽은 파일의 `cache_check` 값입니다.

## 정규화 결함의 재현과 수정

변경 기록 예시　결함 Q98-N01은 “장비A”를 장비 이름으로 읽지 못했습니다. 변경은 A·B 뒤에 더 이어지는 영문·숫자가 없을 때만 장비와 글자 사이에 공백을 넣은 한 규칙입니다. Q98-G01의 “장비AB”는 그대로 두고 기존 Q01도 유지합니다. 111장에서 새로 발견한 사용자 결함이 아니라 98장의 수정 사례를 후보판으로 다시 확인하는 예시입니다.


### 셀 01 · 기존 정상 입력의 회귀 테스트

후보판의 규칙을 이미 띄어쓰기가 있는 Q01에 다시 적용합니다.

후보 카드에는 검사한 대상을 적으세요. 장비명 공백을 바로잡았다는 출력은 입력 정규화의 근거입니다. 최종 응답의 RULE-A까지 맞는지는 같은 질문을 끝까지 실행한 회귀 기록에서 따로 확인합니다.


In [1]:
import re

def normalize(question):
    return re.sub(r"장비([AB])(?=$|[^A-Za-z0-9])", r"장비 \1", question)

original = "장비 A의 점검 전 확인 사항은?"
print(normalize(original) == original)

True


True입니다. 새 표기를 처리하도록 바꾸면서 기존 입력의 형태까지 바뀌지 않았습니다. 이 한 건과 모호한 이름, 처음 실패한 이름을 따로 기록해야 수정의 범위를 설명할 수 있습니다.

기존 정상 문항의 True는 원래 동작이 유지됐다는 결과입니다. 수정한 결함의 해결 여부는 그 결함을 재현했던 최소 입력으로 확인해야 합니다. 릴리스 후보에는 기존 정상 사례와 수정 대상 사례의 결과를 함께 남깁니다.


## 회귀 통과 범위와 출시 기준

후보 기록　공개 기본 문항 7개는 변경 전후 모두 7개 통과. 공개 확장 3개는 1개에서 3개 통과. 같은 원문과 질문을 사용한 로컬 비교입니다. 실제 사용자 관찰과 HTTP·운영 성능 확인은 남아 있습니다. 다음 팀이 이 결과를 읽을 때 ‘공개 사례에 맞춘 후보’와 ‘배포 승인’을 구분할 수 있어야 합니다.


### 팀 참고 예시 · 최소 실패 입력 재검증


In [2]:
print(normalize("장비A의 점검 전 확인 사항은?"))

장비 A의 점검 전 확인 사항은?


장비 A가 분리되어 나옵니다. 이 문자열을 넘긴 뒤 올바른 RULE-A가 돌아오는지까지 봐야 해당 사용자 경로를 닫을 수 있습니다. 평가 기록 Q98-N01의 후보 답은 “점검 전 전원 표시등을 확인합니다.”이고 첫 출처는 RULE-A, 1쪽 2.1절입니다. 아래 전체 평가 파일에서 같은 행의 answer와 sources를 대조할 수 있습니다.

#### 수정이 끝났다고 판정할 범위

<mark>재현 입력과 이전의 정상 입력을 함께 통과해야 변경 범위를 설명할 수 있습니다.</mark> 공백 없는 이름은 고쳐지고 더 긴 장비명은 보존되는지 확인하세요. 통과한 두 입력을 적으면 검증한 표기와 아직 시험하지 않은 표기를 가를 수 있습니다.

선수 개념 · 98장 정규표현식 치환과 경계 조건

#### 붙여 쓴 장비명을 고친 뒤 기존 Q01을 다시 통과시키기

```mermaid
flowchart TD
nq["기존 Q01"]
  nmatch["패턴 대조"]
  nreplace["공백 삽입"]
  nkeep["문자열 유지"]
  ncompare["기존값 대조"]
  nq --> nmatch
  nmatch --> nreplace
  nmatch --> nkeep
  nkeep --> ncompare
  nreplace --> ncompare
```

이미 공백이 있는 정상 질문을 수정 후보에 넣습니다. 같은 문자열이 그대로 나오는지도 회귀 검사에 포함합니다.

<p><a download="historical-regression.json" href="...">Q98-N01 응답과 출처가 담긴 평가 기록 받기</a></p>


## 후보판을 넘기기 전에 기대 문자열을 세 건으로 고정하기

98장에서 고친 장비명 표기 처리를 후보판에 넣습니다. 가장 짧은 재현 입력 외에도 이미 맞던 입력과 손대지 않을 이름을 한 묶음으로 검사해야 다음 사람이 같은 변경을 검증할 수 있습니다.

### 셀 04 · 예상 문자열과 실제 결과를 건별로 대조하기

기존 normalize 함수에 일반 표기, 붙인 표기, 더 긴 이름을 넣습니다. 세 가지 중 하나라도 바뀌면 어느 입력에서 실패했는지 즉시 확인하게 합니다.

In [3]:
regression = {
    "기존 Q01": ("장비 A의 점검", "장비 A의 점검"),
    "재현 Q98-N01": ("장비A의 점검", "장비 A의 점검"),
    "보존 Q98-G01": ("장비AB의 점검", "장비AB의 점검"),
}
for label, (query, expected) in regression.items():
    actual = normalize(query)
    assert actual == expected, (label, query, expected, actual)
print(f"표기 회귀 {len(regression)}건 통과")

표기 회귀 3건 통과


통과한 세 건은 입력 문자열 변환 사례입니다. 111장의 일곱 공개 질문에 대한 업무 평가와 분모가 다릅니다. 면접이나 발표에서 두 결과를 하나의 성공률로 합치지 않습니다.

### 수정이 없던 기준선에서도 실행하기

별도 사본에서 normalize 자리에 입력을 그대로 돌려주는 함수를 넣으세요. 붙여 쓴 Q98-N01이 실패하고, 오류 메시지에 원문·기대값·실제값이 모두 남는지 확인합니다. 이후 원래 함수를 복원하고 같은 묶음을 다시 실행하세요.

<mark>후보판을 통과시킨 근거에는 무엇을 시험했는지까지 들어가야 합니다.</mark> 문자열 회귀를 통과해도 저장 실패 복구나 HTTP 권한 경계를 시험한 것은 아닙니다. 검사 대상이 다르면 결과표를 분리합니다.

112장 배포 인계에는 이 후보의 식별자와 테스트 범위, 아직 비어 있는 배포 증거를 함께 적습니다.

## 팀 사례에 적용하기 · 캐시와 표기 수정의 회귀 결과를 묶기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
replay = [normalize(row["input"]) == row["expected"] for row in incoming["checks"]]
ok = all(replay) and incoming["cache_check"]["same"]
new = {**incoming, "regression": {"spelling": replay, "local_candidate_ok": ok}}
print(replay, ok)

[True, True] True


공백을 없앤 장비 A만 고치고 장비AB를 보존해야 두 표기 검사가 함께 통과합니다. cache_check도 같은 팀의 결과이므로 캐시를 바꾸면 다시 실행하세요.

In [ ]:
output_file = project_root / "team-project/final/chapter111.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter111.json


112장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

112장에 넘길 후보에는 코드·문서·질문 판본, 수정 내용, 회귀 결과와 보류한 배포 조건이 함께 있어야 합니다. 이 로컬 후보의 배포 결정은 hold_pending_evidence입니다.
